# AdaptAuth-ML v11: demostración de ejecución

**Modelo de autenticación adaptativa basado en Machine Learning para la detección de accesos anómalos en PYMES del sector de servicios empresariales en Lima, 2026.**

Tesista: Ashly Herminia Jasmín Chávez Marón · Asesor: Ing. Daniel Alfonso Quinto Pazce · UNMSM, Tesis II

Código fuente: [github.com/AshlyChavez/adaptauth-ml](https://github.com/AshlyChavez/adaptauth-ml)

**Versión 11.** El motor de detección es un clasificador supervisado (HistGradientBoosting) con decisión de tres niveles, explicación SHAP y reentrenamiento protegido. Isolation Forest se evaluó y se retiró del núcleo porque no mejora el F1 ni el AUC-PR, ni siquiera con pocas etiquetas (paso 5). El umbral de bloqueo θ2 se calibra con FPR ≤ 0.1 %.

| Paso | Qué se ejecuta | Tiempo aproximado |
|---|---|---|
| 1 | Descarga del código desde GitHub | segundos |
| 2 | Dataset SuspiciousLogin | segundos |
| 3 | Líneas base: reglas estáticas e Isolation Forest | 1 a 2 min |
| 4 | AdaptAuth-ML v11, ablación, pruebas pareadas y bootstrap | 3 a 6 min |
| 5 | Arranque en frío y recalibración de θ2 | 2 a 3 min |
| 6 | Verificación de requisitos no funcionales | 2 a 4 min |
| 7 | CERT r4.2: historial del empleado | 10 a 20 min |

Para ejecutar todo: **Entorno de ejecución → Ejecutar todo**.

## Paso 1. Descarga del código desde GitHub

In [ ]:
import os
os.chdir("/content")
!rm -rf adaptauth-ml
!git clone -q https://github.com/AshlyChavez/adaptauth-ml.git
os.chdir("/content/adaptauth-ml")
!pip -q install shap
!git log --oneline -1

## Paso 2. Dataset SuspiciousLogin (Google Workspace)

Inicios de sesión exitosos de dos instituciones durante ocho meses. La etiqueta es la marca de sospecha del motor de riesgo de Google Workspace, no un ataque confirmado.

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.width", 200)
if not os.path.exists("/content/SL"):
    !git clone -q --depth 1 https://github.com/salomaopena/SuspiciousLoginDataset.git /content/SL
SL = glob.glob("/content/SL/**/suspicious_logins_public_v1.csv", recursive=True)[0]
d = pd.read_csv(SL)
print(f"Eventos: {len(d):,} | Usuarios: {d['actor_pseudo_id'].nunique():,} | Sospechosos: {d['label'].mean()*100:.2f} %")
display(d[["event_hour", "ip_country", "new_ip", "is_new_country", "impossible_travel", "distinct_ips_30d", "logins_30d", "label"]].head())

## Paso 3. Líneas base

Reglas estáticas (IP nueva, país nuevo, viaje imposible, hora atípica) e Isolation Forest aislado. Partición temporal: entrenamiento de diciembre a marzo, calibración en abril y prueba en mayo y junio.

In [ ]:
!python run_suspiciouslogin.py --data "{SL}"

## Paso 4. AdaptAuth-ML v11 y ablación

Se compara AdaptAuth-ML v11 con el modelo apilado de la versión 10 (Isolation Forest como variable del clasificador).

In [ ]:
!python run_analisis_complementario.py --data "{SL}"

In [ ]:
display(Markdown("### Comparación en prueba (meta: F1 ≥ 0.60)"))
display(pd.read_csv("results_complementarios/ablacion_sl.csv").round(3))
display(Image("results_complementarios/curva_pr_modelos_sl.png", width=800))
j = json.load(open("results_complementarios/analisis_complementario.json"))
display(Markdown("### Pruebas pareadas sobre 14 grupos y bootstrap por usuario"))
display(pd.DataFrame([{"comparación": k, "prueba": v["prueba"], "estadístico": round(v["estadistico"], 2),
                       "p": f'{v["p_valor"]:.3g}', "Shapiro p": round(v["shapiro_p_diferencias"], 3)}
                      for k, v in j["pareadas_14_grupos"].items()]))
display(pd.DataFrame([{"comparación": k, "dif. media de F1": round(v["dif_media"], 3),
                       "IC 95 %": [round(x, 3) for x in v["ic95"]]} for k, v in j["bootstrap_usuarios"].items()]))

## Paso 5. Arranque en frío y recalibración de θ2

**Arranque en frío:** se simula una PYME con pocas etiquetas: solo una fracción de los usuarios de entrenamiento conserva su etiqueta. Isolation Forest se ajusta con todos los eventos, porque no necesita etiquetas.

**θ2:** se compara el umbral de bloqueo con distintas restricciones; la versión 11 usa FPR de bloqueo ≤ 0.1 %.

In [ ]:
!python run_levantamiento.py --data "{SL}"

In [ ]:
display(Image("results_levantamiento/arranque_en_frio_sl.png", width=760))
display(pd.read_csv("results_levantamiento/arranque_en_frio_diferencias.csv").round(4))
display(Markdown("### Matriz de decisiones en prueba según la regla de θ2"))
display(pd.read_csv("results_levantamiento/recalibracion_theta2_sl.csv").round(4))

## Paso 6. Verificación de requisitos no funcionales

RNF-04 y RNF-05 en la misma corrida de 1 000 eventos; contingencia, registro, explicación SHAP, reentrenamiento y privacidad.

In [ ]:
!python run_requisitos_sl.py --data "{SL}"

In [ ]:
display(pd.read_csv("results_requisitos_sl/matriz_cumplimiento_requisitos.csv"))
reg = pd.read_csv("results_requisitos_sl/registro_eventos_prueba.csv")
display(pd.crosstab(reg["decision"], reg["etiqueta"].map({0: "legítimo", 1: "sospechoso"})))
display(pd.read_csv("results_requisitos_sl/explicacion_alertas_top3.csv").sort_values("score_R", ascending=False).head(8))

## Paso 7. CERT Insider Threat r4.2: historial del empleado

Organización sintética de 1 000 empleados (Carnegie Mellon University). Validación cruzada de 5 grupos de empleados. Los datos se leen desde **Mi unidad → AdaptAuth-ML → CERT_r4.2** y los resultados se copian a **Mi unidad → AdaptAuth-ML → resultados_v11**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CERT = "/content/drive/MyDrive/AdaptAuth-ML/CERT_r4.2"
!python run_cert_v2.py --dir "{CERT}" --frio
!mkdir -p /content/drive/MyDrive/AdaptAuth-ML/resultados_v11
!cp -r results_cert_v11 results_levantamiento /content/drive/MyDrive/AdaptAuth-ML/resultados_v11/

In [ ]:
c = pd.read_csv("results_cert_v11/tabla_comparativa_cert_v2.csv")
c["modelo"] = c["modelo"].replace({"supervisado": "AdaptAuth-ML v11", "hibrido_apilado": "Apilado con IF (v10)",
                                   "hibrido": "Híbrido lineal", "reglas_estaticas": "Reglas estáticas", "iforest": "Isolation Forest"})
display(c[["modelo", "auc_roc", "auc_pr", "f1_theta1", "recall_theta1", "fpr_theta1"]].round(4))
display(Image("results_cert_v11/figuras/puntos_operacion_cert_v11.png", width=760))
display(Image("results_cert_v11/figuras/comparacion_modelos_cert_v2.png", width=760))
display(pd.read_csv("results_cert_v11/arranque_en_frio_cert.csv").round(4))

## Resumen

Los resultados de cada paso se generan en esta corrida. Código y resultados: [github.com/AshlyChavez/adaptauth-ml](https://github.com/AshlyChavez/adaptauth-ml)